# Part One: Preparing the Data

- Tom
- November 20, 2026

This note is to share how to use the trained ResNet50V2 CNN of the baseline model to transform data preparing for the experiments in the following layers.

## Import the Necessary Libraries

In [1]:
import numpy as np
import cv2
import pandas as pd
import tensorflow as tf

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input
from tensorflow.keras.applications.resnet_v2 import preprocess_input

## Load Original Video Clips

In [2]:
# Locations of the shared data (relative to this notebook)
DATA_ROOT    = '../data/orig_data/ucf101/'

# Split names used in the feature file names and as dictionary keys
TRAIN      = 'train'
VALIDATION = 'val'
TEST       = 'test'

### _Load Clips Information_

In [3]:
def load_clip_info(data_root, data_type, scope=None):
    info_file = data_root + data_type + '.csv'
    df = pd.read_csv(info_file)
    if scope:
        df = df[df['label'].isin(scope)]

    return df

In [4]:
selected_classes = ['BodyWeightSquats', 'Lunges', 'PushUps', 'WallPushups',
                    'PullUps', 'HandstandPushups', 'JumpingJack', 'JumpRope']
clip_files_info = load_clip_info(DATA_ROOT, TRAIN, selected_classes)
clip_files_info['label'].unique()

<StringArray>
[     'JumpingJack',          'PullUps', 'BodyWeightSquats',
 'HandstandPushups',      'WallPushups',         'JumpRope',
           'Lunges',          'PushUps']
Length: 8, dtype: str

### _Create Label Mapping_

In [5]:
label_dict = {}

for i in range(len(selected_classes)):
    label_dict[selected_classes[i]] = i

label_dict

{'BodyWeightSquats': 0,
 'Lunges': 1,
 'PushUps': 2,
 'WallPushups': 3,
 'PullUps': 4,
 'HandstandPushups': 5,
 'JumpingJack': 6,
 'JumpRope': 7}

### _Read Frames_

In [6]:
def load_clip(clip_path, resize=None, n_frames=16):
    # create video capture entity
    cap = cv2.VideoCapture(clip_path)
    frames = []

    # read frames from the clip
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if resize:
            frame = cv2.resize(frame, resize)

        # Reorder to RGB, which the pretrained CNNs expect
        frame = cv2.cvtColor(frame, cv2.COLOR_BGRA2RGB)

        frames.append(frame)
    # ok, frame = cap.read() # Read the first frame
    # # Read every frame until the video ends
    # while ok:
    #     frame = cv2.resize(frame, (size, size)) # Resize the frame to (224, 224, 3)
    #     frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)) 
    #     ok, frame = cap.read() # Read the next frame
    cap.release() # Close video file
    # Get 32 evenly spaced frames
    indexes = np.linspace(0, len(frames) - 1, n_frames).astype(int)
    # Combine individual frame arrays into one array, keep the 32 chosen frames
    clip = np.stack(frames).astype(np.float32)[indexes]
    return preprocess_input(clip)

In [7]:
def load_clips(clip_path, scope, frame_size=(224, 224), n_frames=16):
    """
    Load all of clips.
    """
    features = {}
    labels = {}

    for name in [TRAIN, VALIDATION, TEST]:
        X = []
        y = []
        clip_info = load_clip_info(clip_path, name, scope)

        # iterate all rows
        for row in clip_info.itertuples(index=False):
            label = label_dict[row.label]
            y.append(label)

            clip_file = clip_path + row.clip_path
            clip = load_clip(clip_file, frame_size, n_frames)

            X.append(clip)

        X = np.stack(X).astype(np.float32)
        y = np.array(y, dtype=np.int32)

        features[name] = X
        labels[name] = y

    return (features, labels)


In [8]:
features, labels = load_clips(DATA_ROOT, selected_classes)

## Transform Data

### _Load Train Model_

In [14]:
# 1. Load your stored model
trained_model = tf.keras.models.load_model("baseline_20261120.keras")
trained_model.summary()

# 2. Extract the ResNet50V2 sub-model by its layer name
resnet_extractor = trained_model.get_layer("base_timedistrubed")

Model: "Baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ base_timedistrubed              │ (None, 16, 2048)       │    23,564,800 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ base_lstm (LSTM)                │ (None, 256)            │     2,360,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 8)              │         2,056 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 77,690,650 (296.37 MB)

 Trainable params: 25,881,736 (98.73 MB)

 Non-trainable params: 45,440 (177.50 KB)

 Optimizer params: 51,763,474 (197.46 MB)

### _Create a New Model Using the Extracted ResNet50V2 Layer_

In [10]:
input_shape = features[TRAIN].shape[1:]

model = Sequential()
model.add(Input(shape=input_shape))
model.add(resnet_extractor)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ base_timedistrubed              │ (None, 16, 2048)       │    23,564,800 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,564,800 (89.89 MB)

 Trainable params: 23,519,360 (89.72 MB)

 Non-trainable params: 45,440 (177.50 KB)

### _Transform Features_

In [11]:
# Suppose `video_clips` is a large array, memory-mapped array, or generator of chunks
def encode_clips(model, video_clips, batch_size):
    feature_chunks = []

    for i in range(0, len(video_clips), batch_size):
        chunk = video_clips[i : i + batch_size]
        # Preprocess if not already handled inside your model
        # chunk = keras.applications.resnet_v2.preprocess_input(chunk.astype("float32"))
        # Extract features using your wrapped TimeDistributed model
        features = model.predict(chunk)
        feature_chunks.append(features)

    return np.concatenate(feature_chunks, axis=0)

In [12]:
for name in [TRAIN, VALIDATION, TEST]:
    X_encoded = encode_clips(model, features[name], 32)
    file_name = f'./dataset/resnet50v2_224_{name}.npy'
    np.save(file_name, X_encoded)

1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 191ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 11s 11s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s

### _Save Labels_

In [13]:
# Save features + unpack all 3 label sets as separate arrays
np.savez("label_dict.npz", **labels)